# 03 — MedGemma Fine-tuning for Skin Analysis

Fine-tune Google's MedGemma medical vision model using LoRA/QLoRA.

**Requirements:** Colab with T4 or A100 GPU
**Input:** Labeled images from `01_data_preparation.ipynb`
**Output:** LoRA adapter weights on Google Drive / HuggingFace Hub

In [ ]:
!pip install -q transformers peft datasets accelerate bitsandbytes pillow tqdm scikit-learn huggingface_hub

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, json, torch

DATA_DIR = '/content/drive/MyDrive/RadianceIQ/datasets/processed'
OUTPUT_DIR = '/content/drive/MyDrive/RadianceIQ/models/medgemma'
os.makedirs(OUTPUT_DIR, exist_ok=True)

HF_TOKEN = userdata.get('HF_TOKEN')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

## 1. Load MedGemma with QLoRA

In [ ]:
from transformers import AutoModelForCausalLM, AutoProcessor, BitsAndBytesConfig, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = 'google/medgemma-4b-it'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map='auto',
    trust_remote_code=True, token=HF_TOKEN,
)
processor = AutoProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)
print(f'Model loaded: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M params')

## 2. Configure LoRA

In [ ]:
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)
model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable/1e6:.1f}M / {total/1e6:.1f}M ({100*trainable/total:.2f}%)')

## 3. Prepare Dataset

In [ ]:
from torch.utils.data import Dataset
from PIL import Image

PROMPT = 'Analyze this skin photo. Return JSON: {"acne_score": 0-100, "sun_damage_score": 0-100, "skin_age_score": 0-100, "confidence": "low"|"med"|"high", "primary_driver": str, "recommended_action": str}'

class SkinDataset(Dataset):
    def __init__(self, data_path, processor, max_length=512):
        with open(data_path) as f:
            self.data = json.load(f)
        self.processor = processor
        self.max_length = max_length
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        try:
            image = Image.open(item['image_path']).convert('RGB').resize((512, 512))
        except Exception:
            image = Image.new('RGB', (512, 512), (128, 128, 128))
        
        scores = {k: item[k] for k in ['acne_score', 'sun_damage_score', 'skin_age_score']}
        primary = max(scores, key=scores.get)
        target = json.dumps({**scores, 'confidence': 'high', 'primary_driver': primary, 'recommended_action': f'Focus on {primary.replace("_", " ")} management.'})
        
        inputs = self.processor(text=f'{PROMPT}\n{target}', images=image, return_tensors='pt', padding='max_length', max_length=self.max_length, truncation=True)
        return {k: v.squeeze(0) for k, v in inputs.items()}

train_dataset = SkinDataset(f'{DATA_DIR}/train_augmented.json', processor)
val_dataset = SkinDataset(f'{DATA_DIR}/val.json', processor)
print(f'Train: {len(train_dataset)}, Val: {len(val_dataset)}')

## 4. Train

In [ ]:
import time

training_args = TrainingArguments(
    output_dir=f'{OUTPUT_DIR}/checkpoints', num_train_epochs=3,
    per_device_train_batch_size=2, per_device_eval_batch_size=2,
    gradient_accumulation_steps=4, learning_rate=2e-4, weight_decay=0.01,
    warmup_steps=50, logging_steps=10, eval_strategy='steps', eval_steps=50,
    save_strategy='steps', save_steps=100, save_total_limit=3,
    fp16=True, report_to='none', dataloader_pin_memory=False,
)

trainer = Trainer(model=model, args=training_args, train_dataset=train_dataset, eval_dataset=val_dataset)

start_time = time.time()
trainer.train()
training_minutes = (time.time() - start_time) / 60
print(f'Training completed in {training_minutes:.1f} minutes')

## 5. Save Adapter Weights

In [ ]:
adapter_path = f'{OUTPUT_DIR}/adapter'
model.save_pretrained(adapter_path)
processor.save_pretrained(adapter_path)

with open(f'{OUTPUT_DIR}/training_info.json', 'w') as f:
    json.dump({'base_model': MODEL_ID, 'training_time_minutes': training_minutes, 'lora_r': 16, 'lora_alpha': 32, 'epochs': 3, 'train_samples': len(train_dataset), 'val_samples': len(val_dataset)}, f, indent=2)

print(f'Adapter saved to: {adapter_path}')

## 6. Push to HuggingFace Hub (Optional)

In [ ]:
# from huggingface_hub import login
# HF_USERNAME = 'your-username'
# login(token=HF_TOKEN)
# model.push_to_hub(f'{HF_USERNAME}/radianceiq-medgemma-lora')
# processor.push_to_hub(f'{HF_USERNAME}/radianceiq-medgemma-lora')

## 7. Test Inference

In [ ]:
import re
with open(f'{DATA_DIR}/test.json') as f:
    test_data = json.load(f)

test_image = Image.open(test_data[0]['image_path']).convert('RGB').resize((512, 512))
inputs = processor(text=PROMPT, images=test_image, return_tensors='pt').to(model.device)

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=300, temperature=0.2, do_sample=True)

result = processor.decode(outputs[0], skip_special_tokens=True)
json_match = re.search(r'\{[^}]+\}', result)
if json_match:
    print('Model output:', json.dumps(json.loads(json_match.group()), indent=2))
else:
    print('Raw output:', result)

print(f'Ground truth: acne={test_data[0]["acne_score"]}, sun={test_data[0]["sun_damage_score"]}, age={test_data[0]["skin_age_score"]}')